# Complete Glaucoma MLDL Colab Pipeline

Run in Colab with a T4 GPU. This master notebook audits the datasets, reproduces partial baselines, trains the pooled fine-tuned classifier, trains segmentation baselines, generates Grad-CAM and visual artifacts, validates the measured export, and optionally copies outputs to Google Drive.

In [ ]:
# 1. Project setup
from pathlib import Path
import os, sys

PROJECT_ROOT = Path('/content/mldl_project')
if not PROJECT_ROOT.exists():
    raise RuntimeError('Upload or clone mldl_project at /content/mldl_project before running.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)


In [ ]:
# 2. Verify GPU and packages
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
!python -m pip install -q joblib scikit-learn matplotlib pillow openpyxl rarfile


In [ ]:
# 3. Verify the required raw archives exist
from pathlib import Path
required = [
    'data/raw/ACRIMA_Database.zip',
    'data/raw/RIM-ONE-r3.zip',
    'data/raw/ORIGA_candidate_figshare.zip',
    'data/raw/REFUGE_figshare.rar',
    'data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip',
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise RuntimeError('Missing required archives: ' + ', '.join(missing))
print('All required archives are present.')


In [ ]:
# 4. Dataset and manifest audits
!python tools/audit_acrima.py
!python tools/audit_rim_one_r3.py
!python tools/audit_origa_refuge.py
!python tools/audit_refuge_original.py
!python tools/audit_drishti_mirror.py
!python tools/audit_drishti_mirror.py


In [ ]:
# 5. Reproduce the existing CPU-compatible baselines
!python tools/train_acrima_baseline.py
!python tools/package_acrima_classifier.py
!python tools/evaluate_acrima_on_rim.py
!python tools/evaluate_acrima_on_repackaged.py
!python tools/train_rim_segmentation.py
!python tools/evaluate_rim_segmentation.py


In [ ]:
# 6. Reproduce the four-dataset pooled baseline
!python tools/train_pooled_frozen_baseline.py
!python tools/evaluate_pooled_refuge_official_test.py
!python tools/compare_classifiers_common_splits.py


In [ ]:
# 7. Train the final fine-tuned pooled classifier
!python tools/train_pooled_finetuned_classifier.py


In [ ]:
# 8. Train segmentation baselines
!python tools/train_pooled_segmentation.py
!python tools/render_pooled_segmentation.py
!python tools/train_drishti_segmentation.py
!python tools/evaluate_drishti_mirror.py


In [ ]:
# 9. Reproduce the matched crop-vs-whole experiment
!python tools/train_rim_whole_vs_crop.py
!python tools/render_rim_crop_comparison.py


In [ ]:
# 10. Generate Grad-CAM and visual artifacts
!python tools/render_gradcam.py
!python tools/render_pooled_gradcam.py
!python tools/render_rim_predictions.py
!python tools/render_baseline_architecture.py


In [ ]:
# 11. Export and validate all completed measured results
!python tools/export_results.py
!python tools/validate_website_export.py
!python tools/audit_split_duplicates.py


In [ ]:
# 12. Package outputs for download
import shutil
from pathlib import Path
package = Path('/content/mldl_project_results')
if package.exists():
    shutil.rmtree(package)
shutil.copytree('outputs', package / 'outputs')
shutil.copytree('data/audit', package / 'data_audit')
shutil.make_archive('/content/mldl_project_results', 'zip', '/content', 'mldl_project_results')
print('/content/mldl_project_results.zip')


In [ ]:
# 13. Optional: copy results to Google Drive
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_out = Path('/content/drive/MyDrive/mldl_project_results')
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2('/content/mldl_project_results.zip', drive_out / 'mldl_project_results.zip')
    print(drive_out / 'mldl_project_results.zip')
